In [1]:
# main_loop_joint.py
# Integrated (joint) matching + rebalancing model with simulation updates

import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import time 

def run_joint(
    FLEET_SIZE=256,
    Penalty=1000.0,
    alpha=50.0,
    Tprime_min=15,
):
    # --------------------------
    # Config (paths)
    # --------------------------
    REQS_CSV   = "test_trips_Jul14_18to20.csv"
    ZONES_CSV  = "taxi_zones_toysample_TableToExcel_csv.csv"
    PHI_CSV    = f"phi_avg_{Tprime_min}min_Jul7to11_18to20.csv"
    # FLEET_SIZE = 256
    REBAL_ONLY_MATCH_REQ = True
    SEED  = 42
    BIG_M = 1e9
    # Tprime_sec = 15 * 60.0
    # Tprime_sec = Tprime_min * 60.0
    solver_time = 0.0
    # for converting Euclidean distance → minutes (for rebalancing legs)
    AVG_SPEED_KMPH = 25.0
    AVG_SPEED_M_PER_MIN = AVG_SPEED_KMPH * 1000.0 / 60.0

    rng = np.random.default_rng(SEED)

    # --------------------------
    # Load trips (already filtered test set)
    # --------------------------
    df = pd.read_csv(REQS_CSV)

    df["request_datetime"]  = pd.to_datetime(df["request_datetime"],  errors="coerce")
    df["on_scene_datetime"] = pd.to_datetime(df["on_scene_datetime"], errors="coerce")  ### NEW
    df["pickup_datetime"]   = pd.to_datetime(df["pickup_datetime"],   errors="coerce")
    df["dropoff_datetime"]  = pd.to_datetime(df["dropoff_datetime"],  errors="coerce")

    df = df.dropna(subset=["request_datetime"]).copy()

    # Normalize column name references
    cols = {c.lower(): c for c in df.columns}
    PU_ID, DO_ID = cols["pulocationid"], cols["dolocationid"]
    PU_X, PU_Y   = cols["pu_x"], cols["pu_y"]
    DO_X, DO_Y   = cols["do_x"], cols["do_y"]

    # Sort & minute bins (for batching)
    df = df.sort_values("request_datetime").reset_index(drop=True)
    df["minute_bin"] = df["request_datetime"].dt.floor("min")

    # --------------------------
    # Durations from REAL timestamps
    # --------------------------
    # 1) driver travel to pickup (deadheading)
    df["wait_min"] = np.ceil(
        (df["on_scene_datetime"] - df["request_datetime"]).dt.total_seconds() / 60.0
    )

    # 2) boarding / curb wait (driver is at pickup, passenger boarding) - time only, no VMT
    df["board_min"] = np.ceil(
        (df["pickup_datetime"] - df["on_scene_datetime"]).dt.total_seconds() / 60.0
    )

    # 3) passenger on-board travel
    df["trip_min"] = np.ceil(
        (df["dropoff_datetime"] - df["pickup_datetime"]).dt.total_seconds() / 60.0
    )

    # 4) NEW: total service time from request → dropoff (for impact / φ-γ balance)
    df["total_serv_min"] = df["wait_min"] + df["board_min"] + df["trip_min"]  ### NEW

    # Clean invalids: enforce at least 1 minute for any nonpositive/missing values
    for col in ["wait_min", "board_min", "trip_min", "total_serv_min"]:
        df.loc[(df[col].isna()) | (df[col] <= 0), col] = 1.0

    # --------------------------
    # Zones (as-is)
    # --------------------------
    zones = (
        pd.read_csv(ZONES_CSV)
        [["LocationID", "centroid_x", "centroid_y"]]
        .drop_duplicates()
        .rename(columns={"LocationID": "zone"})
    )
    zones["zone"] = zones["zone"].astype(int)
    zones = zones.sort_values("zone").reset_index(drop=True)

    zone_ids     = zones["zone"].tolist()
    Z            = len(zone_ids)
    zone_to_idx  = {z: i for i, z in enumerate(zone_ids)}
    zones_mat    = zones.set_index("zone").loc[zone_ids, ["centroid_x", "centroid_y"]].to_numpy(float)

    # --------------------------
    # φ (avg demand per zone × 15-min bin)
    # --------------------------
    phi_tbl = pd.read_csv(PHI_CSV)
    if "zone" not in phi_tbl.columns and "pulocationid" in phi_tbl.columns:
        phi_tbl = phi_tbl.rename(columns={"pulocationid": "zone"})
    phi_tbl["zone"] = phi_tbl["zone"].astype(int)

    def fifteen_bin_label(ts):
        return pd.to_datetime(ts).floor("15min").strftime("%H:%M")

    def phi_vector(tstamp):
        lbl = fifteen_bin_label(tstamp)
        s = phi_tbl.loc[phi_tbl["time_bin_label"] == lbl].set_index("zone")["avg_requests"]
        return np.array([float(s.get(z, 0.0)) for z in zone_ids], dtype=float)

    # --------------------------
    # Initialize fleet evenly (centroid jitter for uniqueness)
    # --------------------------
    per_zone = [FLEET_SIZE // Z] * Z
    for i in range(FLEET_SIZE % Z):
        per_zone[i] += 1

    vehicles = []
    vid = 0
    for zid, n in zip(zone_ids, per_zone):
        cx, cy = zones_mat[zone_to_idx[zid]]
        for _ in range(n):
            vehicles.append(dict(
                id=vid,
                x=cx + rng.normal(0, 20.0),
                y=cy + rng.normal(0, 20.0),
                zone=zid,
                state="idle",   # idle | rebalancing | enroute | ontrip
            ))
            vid += 1
    vehicles = pd.DataFrame(vehicles)

    # ---- add task fields (extended to support on_trip) ----
    vehicles["task_type"]          = None      # "to_pickup" | "on_trip" | "rebalancing" | None(Iddle)
    vehicles["task_remaining_min"] = 0.0
    vehicles["task_request_id"]    = -1
    vehicles["task_dest_x"]        = vehicles["x"]
    vehicles["task_dest_y"]        = vehicles["y"]
    vehicles["task_dest_zone"]     = vehicles["zone"]

    # --- fields for the passenger leg (pickup → dropoff including boarding)
    vehicles["trip_leg_min"]       = 0.0       ### NEW
    vehicles["trip_dest_x"]        = vehicles["x"]
    vehicles["trip_dest_y"]        = vehicles["y"]
    vehicles["trip_dest_zone"]     = vehicles["zone"]

    # --------------------------
    # Helpers for dynamics
    # --------------------------
    def advance_vehicle_states(veh_df, dt_min=1.0):
        """
        Advance ongoing tasks by dt_min and complete any that finish.
        - to_pickup  --> on_trip (start passenger leg)
        - on_trip or rebalancing --> idle
        """
        active_mask = veh_df["task_remaining_min"] > 0
        veh_df.loc[active_mask, "task_remaining_min"] -= dt_min

        done_mask = (veh_df["task_remaining_min"] <= 0) & active_mask

        # vehicles that just finished "to_pickup" (driver reached passenger)
        pickup_done = done_mask & (veh_df["task_type"] == "to_pickup")   ### NEW
        # vehicles that just finished "on_trip" or "rebalancing"
        trip_or_rebal_done = done_mask & (veh_df["task_type"] != "to_pickup")   ### NEW

        # --- 1) handle pickups finishing: start on-trip leg ---
        if pickup_done.any():
            # snap to pickup location (task_dest_*)
            veh_df.loc[pickup_done, "x"]    = veh_df.loc[pickup_done, "task_dest_x"]
            veh_df.loc[pickup_done, "y"]    = veh_df.loc[pickup_done, "task_dest_y"]
            dest_zone = veh_df.loc[pickup_done, "task_dest_zone"]
            veh_df.loc[pickup_done, "zone"] = dest_zone.where(
                dest_zone.notna(),
                veh_df.loc[pickup_done, "zone"]
            )

            # start passenger leg (boarding + in-vehicle time)
            veh_df.loc[pickup_done, "state"]           = "ontrip"
            veh_df.loc[pickup_done, "task_type"]       = "on_trip"
            veh_df.loc[pickup_done, "task_remaining_min"] = veh_df.loc[pickup_done, "trip_leg_min"]

        # --- 2) handle trips or rebalancing finishing: vehicle becomes idle ---
        if trip_or_rebal_done.any():
            veh_df.loc[trip_or_rebal_done, "x"]    = veh_df.loc[trip_or_rebal_done, "task_dest_x"]
            veh_df.loc[trip_or_rebal_done, "y"]    = veh_df.loc[trip_or_rebal_done, "task_dest_y"]
            dest_zone = veh_df.loc[trip_or_rebal_done, "task_dest_zone"]
            veh_df.loc[trip_or_rebal_done, "zone"] = dest_zone.where(
                dest_zone.notna(),
                veh_df.loc[trip_or_rebal_done, "zone"]
            )

            veh_df.loc[trip_or_rebal_done, "state"]           = "idle"
            veh_df.loc[trip_or_rebal_done, "task_type"]       = None
            veh_df.loc[trip_or_rebal_done, "task_request_id"] = -1

        # clean up any negative times
        veh_df.loc[veh_df["task_remaining_min"] < 0, "task_remaining_min"] = 0.0

        return veh_df

    def set_vehicle_to_pickup(veh_df, veh_id, req_idx, Rt):
        """Assign vehicle to go to pickup for request req_idx."""
        veh_df.at[veh_id, "state"]           = "enroute"
        veh_df.at[veh_id, "task_type"]       = "to_pickup"
        veh_df.at[veh_id, "task_request_id"] = int(req_idx)

        # pickup location (this is where the "to_pickup" leg ends)
        veh_df.at[veh_id, "task_dest_x"]     = Rt.iloc[req_idx][PU_X]
        veh_df.at[veh_id, "task_dest_y"]     = Rt.iloc[req_idx][PU_Y]
        veh_df.at[veh_id, "task_dest_zone"]  = Rt.iloc[req_idx][PU_ID]

        # time to get from current location to pickup: USE REAL "driver travel" time
        veh_df.at[veh_id, "task_remaining_min"] = float(Rt.iloc[req_idx]["wait_min"])

        # passenger leg (boarding + in-vehicle) to dropoff
        trip_leg = float(Rt.iloc[req_idx]["board_min"] + Rt.iloc[req_idx]["trip_min"])   ### NEW
        veh_df.at[veh_id, "trip_leg_min"]    = trip_leg
        veh_df.at[veh_id, "trip_dest_x"]     = Rt.iloc[req_idx][DO_X]
        veh_df.at[veh_id, "trip_dest_y"]     = Rt.iloc[req_idx][DO_Y]
        veh_df.at[veh_id, "trip_dest_zone"]  = Rt.iloc[req_idx][DO_ID]

    def set_vehicle_to_rebalance(veh_df, veh_id, z_local):
        """Assign vehicle to rebalance to zone with index z_local in zone_ids / zones_mat."""
        cx, cy        = zones_mat[z_local]
        dest_zone_id  = zone_ids[z_local]

        veh_df.at[veh_id, "state"]     = "rebalancing"
        veh_df.at[veh_id, "task_type"] = "rebalancing"
        veh_df.at[veh_id, "task_dest_x"]    = cx
        veh_df.at[veh_id, "task_dest_y"]    = cy
        veh_df.at[veh_id, "task_dest_zone"] = dest_zone_id
        # task_remaining_min is set later using distance / speed

    # --------------------------
    # Metrics accumulators
    # --------------------------
    total_idle_vmt      = 0.0   # deadheading to pickup (both idle→req, rebal→req)
    total_rebal_vmt     = 0.0   # rebalancing distance
    total_served_reqs   = 0
    total_unserved_reqs = 0

        # --------------------------
    # Main loop: per-minute batches → build inputs
    # --------------------------
    unique_minutes = df["minute_bin"].dropna().sort_values().unique()
    # print(f"Total unique minutes with requests: {len(unique_minutes)}")
    epochs = []

    for t in unique_minutes[:]:
        # 0) advance ongoing tasks
        vehicles = advance_vehicle_states(vehicles, dt_min=1.0)

        Rt = df.loc[df["minute_bin"] == t].copy()
        R  = len(Rt)
        if R == 0:
            continue

        # vehicle sets after advancement
        idx_idle  = vehicles.index[vehicles["state"] == "idle"].to_list()
        idx_rebal = vehicles.index[vehicles["state"] == "rebalancing"].to_list()
        VI, VB    = len(idx_idle), len(idx_rebal)

        # unified vehicle index list for this epoch
        Vstar_idx = idx_idle + idx_rebal          # actual df indices
        Vstar     = len(Vstar_idx)
        J         = R + Z

        # mapping: vehicle id (df index) → row in c3/impact
        v_to_row = {veh_id: k for k, veh_id in enumerate(Vstar_idx)}

        # --- c1: idle vehicles → requests
        if VI > 0:
            v_xy = vehicles.loc[idx_idle, ["x", "y"]].to_numpy(float)
            r_xy = Rt[[PU_X, PU_Y]].to_numpy(float)
            diffs = v_xy[:, None, :] - r_xy[None, :, :]
            c1 = np.sqrt((diffs ** 2).sum(axis=2))
        else:
            c1 = np.zeros((0, R), dtype=float)

        # --- c2: idle vehicles → zone centroids
        if VI > 0:
            v_xy = vehicles.loc[idx_idle, ["x", "y"]].to_numpy(float)
            diffs = v_xy[:, None, :] - zones_mat[None, :, :]
            c2 = np.sqrt((diffs ** 2).sum(axis=2))
        else:
            c2 = np.zeros((0, Z), dtype=float)

        # --- c3: (idle ∪ rebalancing) → (requests ∪ zones)
        if Vstar > 0:
            v_xy = vehicles.loc[Vstar_idx, ["x", "y"]].to_numpy(float)
            c3 = np.zeros((Vstar, J), dtype=float)

            # to requests: cols 0..R-1
            r_xy = Rt[[PU_X, PU_Y]].to_numpy(float)
            diffr = v_xy[:, None, :] - r_xy[None, :, :]
            c3[:, :R] = np.sqrt((diffr ** 2).sum(axis=2))

            # to zones: cols R..R+Z-1
            diffz = v_xy[:, None, :] - zones_mat[None, :, :]
            c3[:, R:] = np.sqrt((diffz ** 2).sum(axis=2))

            if REBAL_ONLY_MATCH_REQ and VB > 0:
                reb_rows = [v_to_row[i] for i in idx_rebal]
                c3[np.ix_(reb_rows, list(range(R, R + Z)))] = BIG_M
        else:
            c3 = np.zeros((0, J), dtype=float)

        # helper to access c3 with your j in R^t ∪ Z (0..R-1 for requests, 0..Z-1 for zones)
        def c3_cost(v, j):
            row = v_to_row[v]
            if j < R:          # request
                col = j
            else:              # zone j in Z_set, but we pass j as zone index directly (0..Z-1)
                col = R + j
            return c3[row, col]

        # --- φ for this epoch
        phi = phi_vector(pd.to_datetime(t))   # (Z,)

        # --- γ: simple proxy = count of idle vehicles per zone
        gamma = np.zeros(Z, dtype=float)
        for i in idx_idle:
            gamma[zone_to_idx[vehicles.at[i, "zone"]]] += 1.0

        # --- impact: (|V*|, |R|+|Z|, |Z|)
        impact = np.zeros((Vstar, J, Z), dtype=float)
        if Vstar > 0:
            v_cur_zone_ids = [vehicles.at[i, "zone"] for i in Vstar_idx]
            v_cur_idx      = np.array([zone_to_idx[z] for z in v_cur_zone_ids], dtype=int)

            if R > 0:
                do_zone_idx = Rt[DO_ID].map(zone_to_idx).to_numpy(int)
                # USE full service time from request → dropoff
                m_j_min = Rt["total_serv_min"].to_numpy(float)   ### NEW
                # Tprime_min = 15.0

                for jj in range(R):
                    impact[np.arange(Vstar), jj, v_cur_idx] += -1.0
                    dur = m_j_min[jj]
                    if dur > Tprime_min:
                        frac = 0.0
                    else:
                        frac = max(0.0, (Tprime_min - dur) / Tprime_min)
                    impact[np.arange(Vstar), jj, do_zone_idx[jj]] += frac

            for jz in range(Z):
                col = R + jz
                impact[np.arange(Vstar), col, v_cur_idx] += -1.0

        def impact_val(v, j, z):
            """Access impact[v,j,z] using v as df index, j as in R_t∪Z_set."""
            row = v_to_row[v]
            if j < R:
                col = j
            else:
                col = R + j
            return float(impact[row, col, z])

        # --- store epoch bundle (optional) ---
        epochs.append(dict(
            t=pd.to_datetime(t),
            request_batch=Rt.copy(),
            vehicles_snapshot=vehicles.copy(),
            idx_idle=idx_idle,
            idx_rebalancing=idx_rebal,
            c1=c1, c2=c2, c3=c3,
            phi=phi, gamma=gamma,
            impact=impact,
            zone_ids=zone_ids,
        ))

        # print(f"[{t}] R={R} |")

        # ==========================
        # Gurobi part: joint model
        # ==========================
        V_I_t  = idx_idle
        V_B_t  = idx_rebal
        R_t    = list(range(R))
        Z_set  = list(range(Z))
        V_all  = list(set(V_I_t) | set(V_B_t))
        J_all  = list(set(R_t)    | set(Z_set))
        P      = Penalty
        # alpha  = alpha

        m = gp.Model("model_joint")
        m.Params.OutputFlag = 0

        x = m.addVars(V_all, J_all, vtype=GRB.BINARY, name="x")
        u = m.addVars(Z_set, lb=0.0, vtype=GRB.CONTINUOUS, name="u")
        w = m.addVars(Z_set, lb=0.0, vtype=GRB.CONTINUOUS, name="w")

        # objective terms (same structure, but using c3_cost and impact_val)
        term_idle = gp.quicksum(c3_cost(v, j) * x[v, j] for v in V_I_t for j in J_all)
        term_rebal = gp.quicksum(c3_cost(v, j) * x[v, j] for v in V_B_t for j in R_t)
        unserved = P * gp.quicksum(1 - gp.quicksum(x[v, j] for v in V_all) for j in R_t)
        imbalance_penalty = alpha * (gp.quicksum(u[z] for z in Z_set) + gp.quicksum(w[z] for z in Z_set))

        m.setObjective(term_idle + term_rebal + unserved + imbalance_penalty, GRB.MINIMIZE)

        # constraints (same sets, using impact_val)
        m.addConstrs((gp.quicksum(x[v, j] for v in V_all) <= 1 for j in J_all))
        m.addConstrs((gp.quicksum(x[v, j] for j in J_all) <= 1 for v in V_all))
        m.addConstrs((
            gamma[z] + gp.quicksum(impact_val(v, j, z) * x[v, j] for v in V_all for j in J_all) - phi[z]
            == w[z] - u[z]
            for z in Z_set
        ))

        t_solve = time.time()
        m.optimize()
        solver_time += time.time() - t_solve
        # ==========================
        # Parse solution + update states
        # ==========================
        idle_to_request  = {}  # originally idle → requests
        idle_to_zone     = {}  # originally idle → zones
        rebal_to_request = {}  # originally rebal → requests
        rebal_unchanged  = []  # originally rebal, not matched to new request

        if m.status == GRB.OPTIMAL:
            # idle vehicles
            for v in V_I_t:
                assigned = False
                for r in R_t:
                    if x[v, r].X > 0.5:
                        idle_to_request[v] = r
                        assigned = True
                        break
                if not assigned:
                    for z in Z_set:
                        if x[v, z].X > 0.5:
                            idle_to_zone[v] = z
                            assigned = True
                            break

            # rebal vehicles
            for v in V_B_t:
                matched_r = None
                for r in R_t:
                    if x[v, r].X > 0.5:
                        matched_r = r
                        break
                if matched_r is None:
                    rebal_unchanged.append(v)
                else:
                    rebal_to_request[v] = matched_r

            new_idx_idle = [v for v in V_I_t if v not in idle_to_request and v not in idle_to_zone]
            idx_rebalancing = list(idle_to_zone.keys()) + list(rebal_unchanged)

        # print("vehicle-request matching:", idle_to_request, rebal_to_request)
        # print("vehicle-zone matching:", idle_to_zone)

        # --- performance metrics + vehicle state updates ---

        # 1) idle / rebal vehicles assigned to requests → deadheading to pickup
        for v, r in idle_to_request.items():
            total_idle_vmt += c3_cost(v, r)
            total_served_reqs += 1
            set_vehicle_to_pickup(vehicles, v, r, Rt)

        for v, r in rebal_to_request.items():
            total_idle_vmt += c3_cost(v, r)
            total_served_reqs += 1
            set_vehicle_to_pickup(vehicles, v, r, Rt)

        # 2) idle vehicles assigned to zones → rebalancing
        for v, z in idle_to_zone.items():
            dist = c3_cost(v, z)
            total_rebal_vmt += dist
            set_vehicle_to_rebalance(vehicles, v, z)
            tmin = max(1.0, np.ceil(dist / AVG_SPEED_M_PER_MIN)) #We need to compute travel time and set task_remaining_min so the simulation knows how long vehicles are busy doing the rebalancing,
            vehicles.at[v, "task_remaining_min"] = tmin

        # 3) rebal_unchanged keep their previous rebalancing tasks (no change)

        # 4) unserved requests in this batch
        batch_unserved = 0
        for r in R_t:
            served_flag = False
            for v in V_all:
                if x[v, r].X > 0.5:
                    served_flag = True
                    break
            if not served_flag:
                batch_unserved += 1
        total_unserved_reqs += batch_unserved

    # --------------------------
    # Summary
    # --------------------------
    # print("=== Joint model summary over simulated horizon ===")
    # print("Total idle VMT to pickup:", total_idle_vmt)
    # print("Total rebalancing VMT:", total_rebal_vmt)
    # print("Total served requests:", total_served_reqs)
    # print("Total unserved requests:", total_unserved_reqs)

    results = {
        "model": "joint",
        "fleet_size": FLEET_SIZE,
        "P": Penalty,
        "alpha": alpha,
        "Tprime_min": Tprime_min,

        "served": int(total_served_reqs),
        "unserved": int(total_unserved_reqs),

        "idle_vmt": float(total_idle_vmt),
        "rebal_vmt": float(total_rebal_vmt),

        "Model solving runtime": float(solver_time),
    }

    return results


In [2]:
# main_loop_sequential_benchmark.py
# Sequential benchmark: matching then rebalancing
# Structured to be as close as possible to main_loop_joint.py
# Only the Gurobi part (models) is different.

import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import time

def run_benchmark(
    FLEET_SIZE=256,
    Penalty=1000.0,
    alpha=50.0,
    Tprime_min=15,
):
    # --------------------------
    # Config (paths)
    # --------------------------
    REQS_CSV   = "test_trips_Jul14_18to20.csv"
    ZONES_CSV  = "taxi_zones_toysample_TableToExcel_csv.csv"
    PHI_CSV    = f"phi_avg_{Tprime_min}min_Jul7to11_18to20.csv"
    # FLEET_SIZE = 256
    REBAL_ONLY_MATCH_REQ = True   # not really used here, but kept for parity
    SEED  = 42
    BIG_M = 1e9
    # Tprime_sec = Tprime_min * 60.0
    solver_time = 0.0
    # for converting Euclidean distance → minutes (for rebalancing legs)
    AVG_SPEED_KMPH = 25.0
    AVG_SPEED_M_PER_MIN = AVG_SPEED_KMPH * 1000.0 / 60.0

    rng = np.random.default_rng(SEED)

    # --------------------------
    # Load trips (already filtered test set)
    # --------------------------
    df = pd.read_csv(REQS_CSV)

    df["request_datetime"]  = pd.to_datetime(df["request_datetime"],  errors="coerce")
    df["on_scene_datetime"] = pd.to_datetime(df["on_scene_datetime"], errors="coerce")
    df["pickup_datetime"]   = pd.to_datetime(df["pickup_datetime"],   errors="coerce")
    df["dropoff_datetime"]  = pd.to_datetime(df["dropoff_datetime"],  errors="coerce")

    df = df.dropna(subset=["request_datetime"]).copy()

    # Normalize column name references
    cols = {c.lower(): c for c in df.columns}
    PU_ID, DO_ID = cols["pulocationid"], cols["dolocationid"]
    PU_X, PU_Y   = cols["pu_x"], cols["pu_y"]
    DO_X, DO_Y   = cols["do_x"], cols["do_y"]

    # Sort & minute bins (for batching)
    df = df.sort_values("request_datetime").reset_index(drop=True)
    df["minute_bin"] = df["request_datetime"].dt.floor("min")

    # --------------------------
    # Durations from REAL timestamps
    # --------------------------
    # 1) driver travel to pickup (deadheading)
    df["wait_min"] = np.ceil(
        (df["on_scene_datetime"] - df["request_datetime"]).dt.total_seconds() / 60.0
    )

    # 2) boarding / curb wait (driver is at pickup, passenger boarding) - time only, no VMT
    df["board_min"] = np.ceil(
        (df["pickup_datetime"] - df["on_scene_datetime"]).dt.total_seconds() / 60.0
    )

    # 3) passenger on-board travel
    df["trip_min"] = np.ceil(
        (df["dropoff_datetime"] - df["pickup_datetime"]).dt.total_seconds() / 60.0
    )

    # 4) total service time from request → dropoff (for impact / φ-γ balance)
    df["total_serv_min"] = df["wait_min"] + df["board_min"] + df["trip_min"]

    # Clean invalids: enforce at least 1 minute for any nonpositive/missing values
    for col in ["wait_min", "board_min", "trip_min", "total_serv_min"]:
        df.loc[(df[col].isna()) | (df[col] <= 0), col] = 1.0

    # --------------------------
    # Zones (as-is)
    # --------------------------
    zones = (
        pd.read_csv(ZONES_CSV)
        [["LocationID", "centroid_x", "centroid_y"]]
        .drop_duplicates()
        .rename(columns={"LocationID": "zone"})
    )
    zones["zone"] = zones["zone"].astype(int)
    zones = zones.sort_values("zone").reset_index(drop=True)

    zone_ids     = zones["zone"].tolist()
    Z            = len(zone_ids)
    zone_to_idx  = {z: i for i, z in enumerate(zone_ids)}
    zones_mat    = zones.set_index("zone").loc[zone_ids, ["centroid_x", "centroid_y"]].to_numpy(float)

    # --------------------------
    # φ (avg demand per zone × 15-min bin)
    # --------------------------
    phi_tbl = pd.read_csv(PHI_CSV)
    if "zone" not in phi_tbl.columns and "pulocationid" in phi_tbl.columns:
        phi_tbl = phi_tbl.rename(columns={"pulocationid": "zone"})
    phi_tbl["zone"] = phi_tbl["zone"].astype(int)

    def fifteen_bin_label(ts):
        return pd.to_datetime(ts).floor("15min").strftime("%H:%M")

    def phi_vector(tstamp):
        lbl = fifteen_bin_label(tstamp)
        s = phi_tbl.loc[phi_tbl["time_bin_label"] == lbl].set_index("zone")["avg_requests"]
        return np.array([float(s.get(z, 0.0)) for z in zone_ids], dtype=float)

    # --------------------------
    # Initialize fleet evenly (centroid jitter for uniqueness)
    # --------------------------
    per_zone = [FLEET_SIZE // Z] * Z
    for i in range(FLEET_SIZE % Z):
        per_zone[i] += 1

    vehicles = []
    vid = 0
    for zid, n in zip(zone_ids, per_zone):
        cx, cy = zones_mat[zone_to_idx[zid]]
        for _ in range(n):
            vehicles.append(dict(
                id=vid,
                x=cx + rng.normal(0, 20.0),
                y=cy + rng.normal(0, 20.0),
                zone=zid,
                state="idle",   # idle | rebalancing | enroute | ontrip
            ))
            vid += 1
    vehicles = pd.DataFrame(vehicles)

    # ---- add task fields (extended to support on_trip) ----
    vehicles["task_type"]          = None      # "to_pickup" | "on_trip" | "rebalancing" | None (idle)
    vehicles["task_remaining_min"] = 0.0
    vehicles["task_request_id"]    = -1
    vehicles["task_dest_x"]        = vehicles["x"]
    vehicles["task_dest_y"]        = vehicles["y"]
    vehicles["task_dest_zone"]     = vehicles["zone"]

    # --- fields for the passenger leg (pickup → dropoff including boarding)
    vehicles["trip_leg_min"]       = 0.0
    vehicles["trip_dest_x"]        = vehicles["x"]
    vehicles["trip_dest_y"]        = vehicles["y"]
    vehicles["trip_dest_zone"]     = vehicles["zone"]

    # --------------------------
    # Helpers for dynamics   (same structure as joint code)
    # --------------------------
    def advance_vehicle_states(veh_df, dt_min=1.0):
        """
        Advance ongoing tasks by dt_min and complete any that finish.
        - to_pickup  --> on_trip (start passenger leg)
        - on_trip or rebalancing --> idle
        """
        active_mask = veh_df["task_remaining_min"] > 0
        veh_df.loc[active_mask, "task_remaining_min"] -= dt_min

        done_mask = (veh_df["task_remaining_min"] <= 0) & active_mask

        # vehicles that just finished "to_pickup" (driver reached passenger)
        pickup_done = done_mask & (veh_df["task_type"] == "to_pickup")
        # vehicles that just finished "on_trip" or "rebalancing"
        trip_or_rebal_done = done_mask & (veh_df["task_type"] != "to_pickup")

        # --- 1) handle pickups finishing: start on-trip leg ---
        if pickup_done.any():
            # snap to pickup location (task_dest_*)
            veh_df.loc[pickup_done, "x"]    = veh_df.loc[pickup_done, "task_dest_x"]
            veh_df.loc[pickup_done, "y"]    = veh_df.loc[pickup_done, "task_dest_y"]
            dest_zone = veh_df.loc[pickup_done, "task_dest_zone"]
            veh_df.loc[pickup_done, "zone"] = dest_zone.where(
                dest_zone.notna(),
                veh_df.loc[pickup_done, "zone"]
            )

            # start passenger leg (boarding + in-vehicle time)
            veh_df.loc[pickup_done, "state"]              = "ontrip"
            veh_df.loc[pickup_done, "task_type"]          = "on_trip"
            veh_df.loc[pickup_done, "task_remaining_min"] = veh_df.loc[pickup_done, "trip_leg_min"]

        # --- 2) handle trips or rebalancing finishing: vehicle becomes idle ---
        if trip_or_rebal_done.any():
            veh_df.loc[trip_or_rebal_done, "x"]    = veh_df.loc[trip_or_rebal_done, "task_dest_x"]
            veh_df.loc[trip_or_rebal_done, "y"]    = veh_df.loc[trip_or_rebal_done, "task_dest_y"]
            dest_zone = veh_df.loc[trip_or_rebal_done, "task_dest_zone"]
            veh_df.loc[trip_or_rebal_done, "zone"] = dest_zone.where(
                dest_zone.notna(),
                veh_df.loc[trip_or_rebal_done, "zone"]
            )

            veh_df.loc[trip_or_rebal_done, "state"]           = "idle"
            veh_df.loc[trip_or_rebal_done, "task_type"]       = None
            veh_df.loc[trip_or_rebal_done, "task_request_id"] = -1

        # clean up any negative times
        veh_df.loc[veh_df["task_remaining_min"] < 0, "task_remaining_min"] = 0.0

        return veh_df

    def set_vehicle_to_pickup(veh_df, veh_id, req_idx, Rt):
        """Assign vehicle to go to pickup for request req_idx."""
        veh_df.at[veh_id, "state"]           = "enroute"
        veh_df.at[veh_id, "task_type"]       = "to_pickup"
        veh_df.at[veh_id, "task_request_id"] = int(req_idx)

        # pickup location (this is where the "to_pickup" leg ends)
        veh_df.at[veh_id, "task_dest_x"]     = Rt.iloc[req_idx][PU_X]
        veh_df.at[veh_id, "task_dest_y"]     = Rt.iloc[req_idx][PU_Y]
        veh_df.at[veh_id, "task_dest_zone"]  = Rt.iloc[req_idx][PU_ID]

        # time to get from current location to pickup: USE REAL "driver travel" time
        veh_df.at[veh_id, "task_remaining_min"] = float(Rt.iloc[req_idx]["wait_min"])

        # passenger leg (boarding + in-vehicle) to dropoff
        trip_leg = float(Rt.iloc[req_idx]["board_min"] + Rt.iloc[req_idx]["trip_min"])
        veh_df.at[veh_id, "trip_leg_min"]    = trip_leg
        veh_df.at[veh_id, "trip_dest_x"]     = Rt.iloc[req_idx][DO_X]
        veh_df.at[veh_id, "trip_dest_y"]     = Rt.iloc[req_idx][DO_Y]
        veh_df.at[veh_id, "trip_dest_zone"]  = Rt.iloc[req_idx][DO_ID]

    def set_vehicle_to_rebalance(veh_df, veh_id, z_local):
        """Assign vehicle to rebalance to zone with index z_local in zone_ids / zones_mat."""
        cx, cy        = zones_mat[z_local]
        dest_zone_id  = zone_ids[z_local]

        veh_df.at[veh_id, "state"]     = "rebalancing"
        veh_df.at[veh_id, "task_type"] = "rebalancing"
        veh_df.at[veh_id, "task_dest_x"]    = cx
        veh_df.at[veh_id, "task_dest_y"]    = cy
        veh_df.at[veh_id, "task_dest_zone"] = dest_zone_id
        # task_remaining_min is set later using distance / speed

    # --------------------------
    # Metrics accumulators
    # --------------------------
    total_idle_vmt      = 0.0   # deadheading to pickup
    total_rebal_vmt     = 0.0   # rebalancing distance
    total_served_reqs   = 0
    total_unserved_reqs = 0

    # --------------------------
    # Main loop: per-minute batches → build inputs
    # --------------------------
    unique_minutes = df["minute_bin"].dropna().sort_values().unique()
    # print(f"Total unique minutes with requests: {len(unique_minutes)}")
    epochs = []

    for t in unique_minutes[:]:   # use [:] for full horizon later
        # 0) advance ongoing tasks
        vehicles = advance_vehicle_states(vehicles, dt_min=1.0)

        Rt = df.loc[df["minute_bin"] == t].copy()
        R  = len(Rt)
        if R == 0:
            continue

        # vehicle sets after advancement
        idx_idle  = vehicles.index[vehicles["state"] == "idle"].to_list()
        idx_rebal = vehicles.index[vehicles["state"] == "rebalancing"].to_list()
        VI, VB    = len(idx_idle), len(idx_rebal)

        # unified vehicle index list for this epoch
        Vstar_idx = idx_idle + idx_rebal          # actual df indices
        Vstar     = len(Vstar_idx)
        J         = R + Z

        # mapping: vehicle id (df index) → row in c1/c2/c3/impact
        idle_to_row = {veh_id: k for k, veh_id in enumerate(idx_idle)}
        v_to_row    = {veh_id: k for k, veh_id in enumerate(Vstar_idx)}

        # --- c1: idle vehicles → requests
        if VI > 0:
            v_xy = vehicles.loc[idx_idle, ["x", "y"]].to_numpy(float)
            r_xy = Rt[[PU_X, PU_Y]].to_numpy(float)
            diffs = v_xy[:, None, :] - r_xy[None, :, :]
            c1 = np.sqrt((diffs ** 2).sum(axis=2))
        else:
            c1 = np.zeros((0, R), dtype=float)

        def c1_cost(v, r):
            """Cost from idle vehicle v to request r (using c1)."""
            row = idle_to_row[v]
            return float(c1[row, r])

        # --- c2: idle vehicles → zone centroids
        if VI > 0:
            v_xy = vehicles.loc[idx_idle, ["x", "y"]].to_numpy(float)
            diffs = v_xy[:, None, :] - zones_mat[None, :, :]
            c2 = np.sqrt((diffs ** 2).sum(axis=2))
        else:
            c2 = np.zeros((0, Z), dtype=float)

        # --- c3: (idle ∪ rebalancing) → (requests ∪ zones)
        # not strictly needed for sequential, but we build it to stay consistent
        if Vstar > 0:
            v_xy = vehicles.loc[Vstar_idx, ["x", "y"]].to_numpy(float)
            c3 = np.zeros((Vstar, J), dtype=float)

            # to requests: cols 0..R-1
            r_xy = Rt[[PU_X, PU_Y]].to_numpy(float)
            diffr = v_xy[:, None, :] - r_xy[None, :, :]
            c3[:, :R] = np.sqrt((diffr ** 2).sum(axis=2))

            # to zones: cols R..R+Z-1
            diffz = v_xy[:, None, :] - zones_mat[None, :, :]
            c3[:, R:] = np.sqrt((diffz ** 2).sum(axis=2))

            if REBAL_ONLY_MATCH_REQ and VB > 0:
                reb_rows = [v_to_row[i] for i in idx_rebal]
                c3[np.ix_(reb_rows, list(range(R, R + Z)))] = BIG_M
        else:
            c3 = np.zeros((0, J), dtype=float)

        # --- φ for this epoch
        phi = phi_vector(pd.to_datetime(t))   # (Z,)

        # --- γ: simple proxy = count of idle vehicles per zone
        gamma = np.zeros(Z, dtype=float)
        for i in idx_idle:
            gamma[zone_to_idx[vehicles.at[i, "zone"]]] += 1.0

        # --- impact: (|V*|, |R|+|Z|, |Z|)
        impact = np.zeros((Vstar, J, Z), dtype=float)
        if Vstar > 0:
            v_cur_zone_ids = [vehicles.at[i, "zone"] for i in Vstar_idx]
            v_cur_idx      = np.array([zone_to_idx[z] for z in v_cur_zone_ids], dtype=int)

            if R > 0:
                do_zone_idx = Rt[DO_ID].map(zone_to_idx).to_numpy(int)
                # USE full service time from request → dropoff
                m_j_min = Rt["total_serv_min"].to_numpy(float)
                # Tprime_min = 15.0

                for jj in range(R):
                    impact[np.arange(Vstar), jj, v_cur_idx] += -1.0
                    dur = m_j_min[jj]
                    if dur > Tprime_min:
                        frac = 0.0
                    else:
                        frac = max(0.0, (Tprime_min - dur) / Tprime_min)
                    impact[np.arange(Vstar), jj, do_zone_idx[jj]] += frac

            for jz in range(Z):
                col = R + jz
                impact[np.arange(Vstar), col, v_cur_idx] += -1.0

        # --- store epoch bundle (optional) ---
        epochs.append(dict(
            t=pd.to_datetime(t),
            request_batch=Rt.copy(),
            vehicles_snapshot=vehicles.copy(),
            idx_idle=idx_idle,
            idx_rebalancing=idx_rebal,
            c1=c1, c2=c2, c3=c3,
            phi=phi, gamma=gamma,
            impact=impact,
            zone_ids=zone_ids,
        ))

        # print(f"[{t}] R={R} |")

        # ==========================
        # Gurobi part: SEQUENTIAL
        # ==========================

        # ---- 1) Matching model (idle vehicles → requests) ----
        V_I_t = idx_idle          # use actual df indices as variable keys
        R_t   = list(range(R))
        P_penalty = Penalty

        m_match = gp.Model("matching_sequential_with_unserved_penalty")
        m_match.Params.OutputFlag = 0

        x = m_match.addVars(V_I_t, R_t, vtype=GRB.BINARY, name="x")

        m_match.setObjective(
            gp.quicksum(c1_cost(v, r) * x[v, r] for v in V_I_t for r in R_t)
            + P_penalty * gp.quicksum(1 - gp.quicksum(x[v, r] for v in V_I_t) for r in R_t),
            GRB.MINIMIZE
        )

        # each request ≤ 1 vehicle, each vehicle ≤ 1 request
        m_match.addConstrs((gp.quicksum(x[v, r] for v in V_I_t) <= 1 for r in R_t))
        m_match.addConstrs((gp.quicksum(x[v, r] for r in R_t) <= 1 for v in V_I_t))

        t_solve_1 = time.time()
        m_match.optimize()
        solver_time += time.time() - t_solve_1
        vehicle_to_request = {v: None for v in V_I_t}
        unserved_r = []

        if m_match.status == GRB.OPTIMAL:
            for v in V_I_t:
                for r in R_t:
                    if x[v, r].X > 0.5:
                        vehicle_to_request[v] = r
                        # metrics: deadheading distance to pickup
                        total_idle_vmt    += c1_cost(v, r)
                        total_served_reqs += 1
                        break

            for r in R_t:
                served_flag = any(x[v, r].X > 0.5 for v in V_I_t)
                if not served_flag:
                    unserved_r.append(r)
                    total_unserved_reqs += 1

        # print("matching results (v → r):",
        #     {v: r for v, r in vehicle_to_request.items() if r is not None})

        # vehicles still idle after matching (eligible for rebalancing)
        idle_after_match = [v for v in V_I_t if vehicle_to_request[v] is None]

        # ---- 2) Rebalancing model (still-idle vehicles → zones) ----
        VI2 = len(idle_after_match)
        if VI2 > 0:
            # mapping from these vehicles to row index in impact_trunc
            reb_to_row = {veh_id: k for k, veh_id in enumerate(idle_after_match)}

            def c2_cost(v, z):
                # IMPORTANT: c2 rows are ordered by idx_idle, so use idle_to_row for distances
                row = idle_to_row[v]
                return float(c2[row, z])

            # impact_truncated: rows = these vehicles, cols = zone choices (R..R+Z-1)
            v_rows = [v_to_row[i] for i in idle_after_match]
            impact_trunc = impact[v_rows, R:, :]  # shape (VI2, Z, Z)

            V_I2_t = idle_after_match
            Z_set  = list(range(Z))
            # alpha  = 50.0

            m_rebal = gp.Model("rebalancing_sequential_with_dev_penalty")
            m_rebal.Params.OutputFlag = 0

            y = m_rebal.addVars(V_I2_t, Z_set, vtype=GRB.BINARY, name="y")
            u = m_rebal.addVars(Z_set, lb=0.0, vtype=GRB.CONTINUOUS, name="u")
            w = m_rebal.addVars(Z_set, lb=0.0, vtype=GRB.CONTINUOUS, name="w")

            m_rebal.setObjective(
                gp.quicksum(c2_cost(v, z) * y[v, z] for v in V_I2_t for z in Z_set)
                + alpha * (gp.quicksum(u[z] for z in Z_set) + gp.quicksum(w[z] for z in Z_set)),
                GRB.MINIMIZE
            )

            # each vehicle rebalances to at most one zone
            m_rebal.addConstrs((gp.quicksum(y[v, z] for z in Z_set) <= 1 for v in V_I2_t))

            # zone-level balance using truncated impact
            m_rebal.addConstrs((
                gamma[z]
                + gp.quicksum(
                    impact_trunc[reb_to_row[v], z_dest, z] * y[v, z_dest]
                    for v in V_I2_t for z_dest in Z_set
                )
                - phi[z] == w[z] - u[z]
                for z in Z_set
            ))

            t_solve_2 = time.time()
            m_rebal.optimize()
            solver_time += time.time() - t_solve_2

            vehicle_to_zone = {v: None for v in V_I2_t}
            if m_rebal.status == GRB.OPTIMAL:
                for v in V_I2_t:
                    for z in Z_set:
                        if y[v, z].X > 0.5:
                            vehicle_to_zone[v] = z
                            dist = c2_cost(v, z)
                            total_rebal_vmt += dist
                            break

            # print("rebalancing results (v → z):",
            #     {v: z for v, z in vehicle_to_zone.items() if z is not None})

            # --- Apply decisions to vehicle states (same style as joint) ---

            # 1) vehicles assigned to requests
            for v, r in vehicle_to_request.items():
                if r is not None:
                    set_vehicle_to_pickup(vehicles, v, r, Rt)

            # 2) vehicles assigned to zones → rebalancing
            for v, z in vehicle_to_zone.items():
                if z is not None:
                    set_vehicle_to_rebalance(vehicles, v, z)
                    dist = c2_cost(v, z)
                    tmin = max(1.0, np.ceil(dist / AVG_SPEED_M_PER_MIN))
                    vehicles.at[v, "task_remaining_min"] = tmin

            # 3) vehicles that remain idle after rebalancing simply stay idle

        else:
            # No vehicles to rebalance; just apply matching decisions
            for v, r in vehicle_to_request.items():
                if r is not None:
                    set_vehicle_to_pickup(vehicles, v, r, Rt)

    # --------------------------
    # Summary
    # --------------------------
    # print("=== Sequential benchmark summary over simulated horizon ===")
    # print("Total idle VMT to pickup:", total_idle_vmt)
    # print("Total rebalancing VMT:", total_rebal_vmt)
    # print("Total served requests:", total_served_reqs)
    # print("Total unserved requests:", total_unserved_reqs)

    results = {
        "model": "sequential",
        "fleet_size": FLEET_SIZE,
        "P": Penalty,
        "alpha": alpha,
        "Tprime_min": Tprime_min,

        "served": int(total_served_reqs),
        "unserved": int(total_unserved_reqs),

        "idle_vmt": float(total_idle_vmt),
        "rebal_vmt": float(total_rebal_vmt),

        "Model solving runtime": float(solver_time),
    }

    return results



In [3]:
BASE = dict(
    FLEET_SIZE=32,
    Penalty=1000.0,
    alpha=50.0,
    Tprime_min=15,
)

# ----------------------------
# Output formatting (TSV for easy copy/paste)
# Requested columns:
# current_value_of_changing_parameter_changing  total_idle_vmt  total_rebal_vmt
# total_served_reqs  total_unserved_reqs  Model_solvingruntime
# We'll include "model" as an extra column so both models fit in one table.
# ----------------------------
def _get_solver_runtime(res: dict) -> float:
    # Accept either key spelling just in case
    if "Model solving runtime" in res:
        return float(res["Model solving runtime"])
    if "Model_solvingruntime" in res:
        return float(res["Model_solvingruntime"])
    raise KeyError("Solver runtime key not found. Expected 'Model solving runtime'.")

def print_header(param_name):
    print(f"{param_name}\tmodel\ttotal_idle_vmt\ttotal_rebal_vmt\ttotal_served_reqs\ttotal_unserved_reqs\tModel_solvingruntime")

def print_row(value, model_name: str, res: dict):
    runtime = _get_solver_runtime(res)
    print(
        f"{value}\t{model_name}\t"
        f"{float(res['idle_vmt']):.6f}\t{float(res['rebal_vmt']):.6f}\t"
        f"{int(res['served'])}\t{int(res['unserved'])}\t"
        f"{runtime:.6f}"
    )

# ----------------------------
# Run both models with a shared config dict
# ----------------------------
def run_both_models(cfg: dict):
    """
    cfg must include: FLEET_SIZE, Penalty, alpha, Tprime_min
    Returns: (res_seq, res_joint)
    """
    res_seq = run_benchmark(
        FLEET_SIZE=cfg["FLEET_SIZE"],
        Penalty=cfg["Penalty"],
        alpha=cfg["alpha"],
        Tprime_min=cfg["Tprime_min"],
    )

    res_joint = run_joint(
        FLEET_SIZE=cfg["FLEET_SIZE"],
        Penalty=cfg["Penalty"],
        alpha=cfg["alpha"],
        Tprime_min=cfg["Tprime_min"],
    )

    return res_seq, res_joint

# ----------------------------
# Sweep one parameter (others fixed to BASE)
# Prints TSV table + returns list of dict rows (optional)
# ----------------------------
def sweep_param(param_name: str, values, base: dict = None):
    """
    Sweeps ONE parameter; all others held fixed to 'base' (defaults to BASE).
    Prints results in a copy/paste TSV table.
    Returns list of dict rows for pandas/CSV if needed.
    """
    base_cfg = (base or BASE).copy()

    print_header(param_name)
    rows = []

    for v in values:
        cfg = base_cfg.copy()
        cfg[param_name] = v

        res_seq = run_benchmark(
            FLEET_SIZE=cfg["FLEET_SIZE"],
            Penalty=cfg["Penalty"],
            alpha=cfg["alpha"],
            Tprime_min=cfg["Tprime_min"],
        )

        # Print rows in the exact copy/paste-friendly format
        print_row(v, "sequential", res_seq)

    for v in values:
        cfg = base_cfg.copy()
        cfg[param_name] = v

        res_joint = run_joint(
            FLEET_SIZE=cfg["FLEET_SIZE"],
            Penalty=cfg["Penalty"],
            alpha=cfg["alpha"],
            Tprime_min=cfg["Tprime_min"],
        )
        print_row(v, "joint",      res_joint)


# ============================================================
# Example usage (run these one by one)
# ============================================================

# 1) Fleet size sweep
sweep_param("FLEET_SIZE", [32, 64,128,256,512])
# fleet_rows = sweep_param("FLEET_SIZE", [128, 192, 256, 320, 384])

# 2) Penalty sweep
sweep_param("Penalty", [100, 500, 1000, 5000, 10000])

# 3) Alpha sweep
sweep_param("alpha", [0, 10, 25, 50, 100, 200])

# 4) Tprime sweep (optional)
sweep_param("Tprime_min", [5, 10, 15, 20, 30])

FLEET_SIZE	model	total_idle_vmt	total_rebal_vmt	total_served_reqs	total_unserved_reqs	Model_solvingruntime
Set parameter Username
Set parameter LicenseID to value 2716496
Academic license - for non-commercial use only - expires 2026-09-30
32	sequential	100270.394387	70.860915	185	306	0.132722
64	sequential	175063.134973	314.430299	334	157	0.375942
128	sequential	190584.143464	1529.160662	459	32	0.755351
256	sequential	178530.336851	4204.529858	481	10	1.036432
512	sequential	178387.943297	10126.619199	481	10	1.588835
32	joint	96083.603999	31.236885	185	306	0.105120
64	joint	168641.096850	28.150606	328	163	0.243234
128	joint	186091.130652	42.225640	457	34	0.424527
256	joint	173794.564314	28.220766	481	10	0.774802
512	joint	172029.797893	23.193568	481	10	1.279248
Penalty	model	total_idle_vmt	total_rebal_vmt	total_served_reqs	total_unserved_reqs	Model_solvingruntime
100	sequential	910.266613	102.345171	12	479	0.373876
500	sequential	45237.546387	88.458273	148	343	0.139015
1000	sequential	1